In [1]:
import pandas as pd
import json
import re

df = pd.read_csv("cleaned_articles.csv")
print(df.shape)
print(df.columns.tolist())
df.head(2)

(805, 15)
['article_id', 'title', 'content', 'text', 'published_at', 'author', 'url', 'source', 'source_country', 'language', 'raw_category', 'pre_extracted_orgs', 'pre_extracted_persons', 'pre_extracted_locations', 'content_word_count']


,article_id,title,content,text,published_at,author,url,source,source_country,language,raw_category,pre_extracted_orgs,pre_extracted_persons,pre_extracted_locations,content_word_count
0,a709ab47e525f2cc55d5b84d6fa989747d94b3d9,"Why Fannie Mae, Freddie Mac, and Redfin Soared...",Shares of Federal National Mortgage Associatio...,"Why Fannie Mae, Freddie Mac, and Redfin Soared...",2023-12-28T00:00:00Z,Steve Symington,https://www.fool.com/investing/2023/12/28/why-...,fool.com,US,english,"['Economy, Business and Finance']",[{'name': 'motley fool shares of federal natio...,"[{'name': 'freddie mac', 'sentiment': 'negativ...",[],628
1,a0fa852bab25f869c478d474a52997485960c3ba,Should You Buy Enbridge Stock for its 7.65% Yi...,Pipeline Most investors are aware of how lucra...,Should You Buy Enbridge Stock for its 7.65% Yi...,2023-12-28T22:06:00Z,VNExplorer,https://vnexplorer.net/should-you-buy-enbridge...,vnexplorer.net,US,english,"['Economy, Business and Finance']","[{'name': 'enbridge', 'sentiment': 'neutral'}]",[],"[{'name': 'united states', 'sentiment': 'none'}]",629


In [2]:
label_df = df[["article_id", "title", "content"]].copy()
print(label_df.shape)
label_df.head(2)

(805, 3)


,article_id,title,content
0,a709ab47e525f2cc55d5b84d6fa989747d94b3d9,"Why Fannie Mae, Freddie Mac, and Redfin Soared...",Shares of Federal National Mortgage Associatio...
1,a0fa852bab25f869c478d474a52997485960c3ba,Should You Buy Enbridge Stock for its 7.65% Yi...,Pipeline Most investors are aware of how lucra...


In [3]:
def truncate_text(text, max_chars=1800):
    text = str(text)
    return text[:max_chars]

label_df["label_input"] = (
    "Title: " + label_df["title"].fillna("").astype(str) +
    "\n\nContent: " + label_df["content"].fillna("").astype(str).apply(truncate_text)
)

label_df[["article_id", "label_input"]].head(1)

,article_id,label_input
0,a709ab47e525f2cc55d5b84d6fa989747d94b3d9,"Title: Why Fannie Mae, Freddie Mac, and Redfin..."


In [24]:
MACRO_TAGS = [
    "inflation",
    "interest_rates",
    "mortgage_interest_rates",
    "monetary_policy",
    "fiscal_policy",
    "recession_growth",
    "regulation",
    "fx_currency",
    "commodities",
    "geopolitics"
]
INDUSTRY_TAGS = [
    "banking",
    "insurance",
    "real_estate",
    "financials",
    "technology",
    "energy",
    "health_care",
    "industrials",
    "consumer",
    "materials",
    "utilities",
    "communication_services"
]

In [45]:
PROMPT_TEMPLATE = """
You are labeling financial news articles.

Return JSON only:
{{
  "macro_tags": [],
  "industry_tags": [],
  "entity_tags": []
}}

Rules:
- macro_tags must be chosen only from this list:
  {macro_tags}
- industry_tags must be chosen only from this list:
  {industry_tags}
- entity_tags should contain named companies, banks, regulators, funds, government bodies, or organizations explicitly mentioned in the article.
- Use lowercase snake_case for macro_tags and industry_tags.
- Keep entity_tags in normal readable names.
- Do not include explanations.
- Do not include any text before or after the JSON.
- If no tag applies, return an empty list for that field.

Article:
{article}
"""

In [11]:
def build_prompt(article_text):
    return PROMPT_TEMPLATE.format(
        macro_tags=", ".join(MACRO_TAGS),
        industry_tags=", ".join(INDUSTRY_TAGS),
        article=article_text
    )

In [12]:
sample_prompt = build_prompt(label_df.iloc[0]["label_input"])
print(sample_prompt[:3000])


You are labeling financial news articles.

Return valid JSON only with this exact schema:
{
  "macro_tags": [],
  "industry_tags": [],
  "entity_tags": []
}

Rules:
- macro_tags must be chosen only from this list:
  inflation, interest_rates, monetary_policy, fiscal_policy, recession_growth, regulation, fx_currency, commodities, geopolitics

- industry_tags must be chosen only from this list:
  banking, insurance, real_estate, financials, technology, energy, health_care, industrials, consumer, materials, utilities, communication_services

- entity_tags should contain named companies, banks, regulators, funds, government bodies, or organizations explicitly mentioned in the article.
- Use lowercase snake_case for macro_tags and industry_tags.
- Keep entity_tags in normal readable names.
- Do not include explanations.
- Do not include any text before or after the JSON.
- If no tag applies, return an empty list for that field.

Article:
Title: Why Fannie Mae, Freddie Mac, and Redfin Soare

In [13]:
import requests

In [14]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "mistral:latest"

In [15]:
def call_ollama(prompt, model=MODEL_NAME):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    
    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()
    
    data = response.json()
    return data["response"]

In [16]:
raw_output = call_ollama(sample_prompt)
print(raw_output)

 {
  "macro_tags": ["mortgage_interest_rates", "monetary_policy"],
  "industry_tags": ["financials", "real_estate"],
  "entity_tags": ["Fannie Mae", "Freddie Mac", "Redfin"]
}


In [19]:
import json
import re

In [20]:
def extract_json_object(text):
    text = str(text).strip()
    
    # Try direct JSON parse first
    try:
        return json.loads(text)
    except:
        pass
    
    # If model adds extra text, try to recover the JSON block
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    
    # Fallback
    return {
        "macro_tags": [],
        "industry_tags": [],
        "entity_tags": [],
        "parse_error": True,
        "raw_output": text
    }

In [26]:
parsed_output = extract_json_object(raw_output)
parsed_output

{'macro_tags': ['consumer'],
 'industry_tags': ['technology'],
 'entity_tags': ['apple', 'bh_photo_video']}

In [22]:
def normalize_labels(result):
    if not isinstance(result, dict):
        result = {}
    
    macro_tags = result.get("macro_tags", [])
    industry_tags = result.get("industry_tags", [])
    entity_tags = result.get("entity_tags", [])
    
    if not isinstance(macro_tags, list):
        macro_tags = []
    if not isinstance(industry_tags, list):
        industry_tags = []
    if not isinstance(entity_tags, list):
        entity_tags = []
    
    # keep only allowed macro and industry tags
    macro_tags = [x for x in macro_tags if x in MACRO_TAGS]
    industry_tags = [x for x in industry_tags if x in INDUSTRY_TAGS]
    
    # clean entity tags
    entity_tags = [str(x).strip() for x in entity_tags if str(x).strip()]
    
    # remove duplicates while preserving order
    macro_tags = list(dict.fromkeys(macro_tags))
    industry_tags = list(dict.fromkeys(industry_tags))
    entity_tags = list(dict.fromkeys(entity_tags))
    
    return {
        "macro_tags": macro_tags,
        "industry_tags": industry_tags,
        "entity_tags": entity_tags
    }

In [25]:
normalized_output = normalize_labels(parsed_output)
normalized_output

{'macro_tags': [],
 'industry_tags': ['technology'],
 'entity_tags': ['apple', 'bh_photo_video']}

In [28]:
macro_tags = normalized_output["macro_tags"]
industry_tags = normalized_output["industry_tags"]
entity_tags = normalized_output["entity_tags"]

print("Macro:", macro_tags)
print("Industry:", industry_tags)
print("Entity:", entity_tags)

Macro: []
Industry: ['technology']
Entity: ['apple', 'bh_photo_video']


In [29]:
#5 articles test
test_5 = label_df.sample(5, random_state=42).copy()
test_5["prompt"] = test_5["label_input"].apply(build_prompt)

print(test_5.shape)
test_5[["article_id", "title"]]

(5, 5)


,article_id,title
192,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...
718,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...
168,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...
522,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound..."
536,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...


In [30]:
results = []

for i, (_, row) in enumerate(test_5.iterrows(), start=1):
    print(f"Processing article {i}/5...")
    
    raw_output = call_ollama(row["prompt"])
    parsed_output = extract_json_object(raw_output)
    normalized_output = normalize_labels(parsed_output)
    
    results.append({
        "article_id": row["article_id"],
        "title": row["title"],
        "raw_output": raw_output,
        "macro_tags": normalized_output["macro_tags"],
        "industry_tags": normalized_output["industry_tags"],
        "entity_tags": normalized_output["entity_tags"]
    })


Processing article 1/5...
Processing article 2/5...
Processing article 3/5...
Processing article 4/5...
Processing article 5/5...


In [31]:
results_df = pd.DataFrame(results)
results_df

,article_id,title,raw_output,macro_tags,industry_tags,entity_tags
0,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...,"{\n ""macro_tags"": [""technology""],\n ""indust...",[],"[consumer, financials]",[apple]
1,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...,"{\n ""macro_tags"": [""inflation""],\n ""industr...",[inflation],[],"[AMC, Cinemark, Regal Cinemas, Alamo Megaplex,..."
2,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[technology],"[Applied Intuition, Embark Technology, Marceil..."
3,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound...","{\n ""macro_tags"": [""none""],\n ""industry_tag...",[],[],[]
4,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...,"{\n ""macro_tags"": [""geopolitics""],\n ""indus...",[geopolitics],[health_care],"[Leon Cooperman, Allianz]"


In [32]:
test_5_labeled = test_5.merge(
    results_df[["article_id", "macro_tags", "industry_tags", "entity_tags"]],
    on="article_id",
    how="left"
)

test_5_labeled[["article_id", "title", "macro_tags", "industry_tags", "entity_tags"]]

,article_id,title,macro_tags,industry_tags,entity_tags
0,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...,[],"[consumer, financials]",[apple]
1,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...,[inflation],[],"[AMC, Cinemark, Regal Cinemas, Alamo Megaplex,..."
2,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...,[],[technology],"[Applied Intuition, Embark Technology, Marceil..."
3,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound...",[],[],[]
4,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...,[geopolitics],[health_care],"[Leon Cooperman, Allianz]"


In [33]:
test_5_labeled.to_csv("silver_test_batch_5.csv", index=False)

In [35]:
#30 articles test
test_30 = label_df.sample(30, random_state=42).copy()
test_30["prompt"] = test_30["label_input"].apply(build_prompt)

print(test_30.shape)
test_30[["article_id", "title"]]

(30, 5)


,article_id,title
192,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...
718,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...
168,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...
522,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound..."
536,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...
791,9b09edeafdc1bb1ac6542e37f2ac3208d5094389,Wharton Professor Jeremy Siegel Says Stocks Co...
765,cb3d9ad151d0c44da42974845bcfd6e6c43223b4,HyperFiber Launches Green Team to Support Inst...
328,b11a31f0e2cb947f5cece113461f2b927c0901c8,Forever Cheer picks Hong Kong as gateway for g...
218,f5ef318e218c131c132bccaacf7255b6e8dd6115,Chris Martin on being environmentally friendly...
788,60b0393c5d31738af3082077a934927fd4824d9f,Guy Fieri Foundation donates $1.2M to Lahaina ...


In [36]:
results_30 = []

for i, (_, row) in enumerate(test_30.iterrows(), start=1):
    print(f"Processing article {i}/30...")
    
    raw_output = call_ollama(row["prompt"])
    parsed_output = extract_json_object(raw_output)
    normalized_output = normalize_labels(parsed_output)
    
    results_30.append({
        "article_id": row["article_id"],
        "title": row["title"],
        "raw_output": raw_output,
        "macro_tags": normalized_output["macro_tags"],
        "industry_tags": normalized_output["industry_tags"],
        "entity_tags": normalized_output["entity_tags"]
    })


Processing article 1/30...
Processing article 2/30...
Processing article 3/30...
Processing article 4/30...
Processing article 5/30...
Processing article 6/30...
Processing article 7/30...
Processing article 8/30...
Processing article 9/30...
Processing article 10/30...
Processing article 11/30...
Processing article 12/30...
Processing article 13/30...
Processing article 14/30...
Processing article 15/30...
Processing article 16/30...
Processing article 17/30...
Processing article 18/30...
Processing article 19/30...
Processing article 20/30...
Processing article 21/30...
Processing article 22/30...
Processing article 23/30...
Processing article 24/30...
Processing article 25/30...
Processing article 26/30...
Processing article 27/30...
Processing article 28/30...
Processing article 29/30...
Processing article 30/30...


In [37]:
results30_df = pd.DataFrame(results_30)
results30_df

,article_id,title,raw_output,macro_tags,industry_tags,entity_tags
0,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...,"{\n ""macro_tags"": [""consumer""],\n ""industry...",[],[technology],"[apple, bh]"
1,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...,"{\n ""macro_tags"": [""fx_currency""],\n ""indus...",[fx_currency],[],"[amc, cinemark, regal_cinemas, alamo_megaplex,..."
2,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[technology],"[Applied Intuition, Embark Technology, Marceil..."
3,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound...","{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],"[real_estate, financials, banking]","[Vingroup, VHM, VRE, VIC, DXG, PDR, DIG, VCG, ..."
4,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[health_care],"[Leon Cooperman, Allianz]"
5,9b09edeafdc1bb1ac6542e37f2ac3208d5094389,Wharton Professor Jeremy Siegel Says Stocks Co...,"{\n ""macro_tags"": [""federal_reserve"", ""inter...",[interest_rates],"[financials, technology]","[Jeremy Siegel, CNBC's Closing Bell, SPDR S&P ..."
6,cb3d9ad151d0c44da42974845bcfd6e6c43223b4,HyperFiber Launches Green Team to Support Inst...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],"[technology, communication_services]",[HyperFiber]
7,b11a31f0e2cb947f5cece113461f2b927c0901c8,Forever Cheer picks Hong Kong as gateway for g...,"{\n ""macro_tags"": [""geopolitics""],\n ""indus...",[geopolitics],"[health_care, technology]","[Forever Cheer Holding, Zhuo Min, University o..."
8,f5ef318e218c131c132bccaacf7255b6e8dd6115,Chris Martin on being environmentally friendly...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[],"[chris_martin, coldplay, ellie_goulding, bbc_r..."
9,60b0393c5d31738af3082077a934927fd4824d9f,Guy Fieri Foundation donates $1.2M to Lahaina ...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[banking],"[guy_fieri, hawaii_restaurant_association, ame..."


In [38]:
test_30_labeled = test_30.merge(
    results30_df[["article_id", "macro_tags", "industry_tags", "entity_tags"]],
    on="article_id",
    how="left"
)

test_30_labeled[["article_id", "title", "macro_tags", "industry_tags", "entity_tags"]]

,article_id,title,macro_tags,industry_tags,entity_tags
0,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...,[],[technology],"[apple, bh]"
1,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...,[fx_currency],[],"[amc, cinemark, regal_cinemas, alamo_megaplex,..."
2,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...,[],[technology],"[Applied Intuition, Embark Technology, Marceil..."
3,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound...",[],"[real_estate, financials, banking]","[Vingroup, VHM, VRE, VIC, DXG, PDR, DIG, VCG, ..."
4,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...,[],[health_care],"[Leon Cooperman, Allianz]"
5,9b09edeafdc1bb1ac6542e37f2ac3208d5094389,Wharton Professor Jeremy Siegel Says Stocks Co...,[interest_rates],"[financials, technology]","[Jeremy Siegel, CNBC's Closing Bell, SPDR S&P ..."
6,cb3d9ad151d0c44da42974845bcfd6e6c43223b4,HyperFiber Launches Green Team to Support Inst...,[],"[technology, communication_services]",[HyperFiber]
7,b11a31f0e2cb947f5cece113461f2b927c0901c8,Forever Cheer picks Hong Kong as gateway for g...,[geopolitics],"[health_care, technology]","[Forever Cheer Holding, Zhuo Min, University o..."
8,f5ef318e218c131c132bccaacf7255b6e8dd6115,Chris Martin on being environmentally friendly...,[],[],"[chris_martin, coldplay, ellie_goulding, bbc_r..."
9,60b0393c5d31738af3082077a934927fd4824d9f,Guy Fieri Foundation donates $1.2M to Lahaina ...,[],[banking],"[guy_fieri, hawaii_restaurant_association, ame..."


In [39]:
test_30_labeled.to_csv("silver_test_batch_30.csv", index=False)

In [40]:
#50 articles test
test_50 = label_df.sample(50, random_state=42).copy()
test_50["prompt"] = test_50["label_input"].apply(build_prompt)

print(test_50.shape)
test_50[["article_id", "title"]]

(50, 5)


,article_id,title
192,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...
718,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...
168,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...
522,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound..."
536,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...
791,9b09edeafdc1bb1ac6542e37f2ac3208d5094389,Wharton Professor Jeremy Siegel Says Stocks Co...
765,cb3d9ad151d0c44da42974845bcfd6e6c43223b4,HyperFiber Launches Green Team to Support Inst...
328,b11a31f0e2cb947f5cece113461f2b927c0901c8,Forever Cheer picks Hong Kong as gateway for g...
218,f5ef318e218c131c132bccaacf7255b6e8dd6115,Chris Martin on being environmentally friendly...
788,60b0393c5d31738af3082077a934927fd4824d9f,Guy Fieri Foundation donates $1.2M to Lahaina ...


In [41]:
results_50 = []

for i, (_, row) in enumerate(test_50.iterrows(), start=1):
    print(f"Processing article {i}/50...")
    
    raw_output = call_ollama(row["prompt"])
    parsed_output = extract_json_object(raw_output)
    normalized_output = normalize_labels(parsed_output)
    
    results_50.append({
        "article_id": row["article_id"],
        "title": row["title"],
        "raw_output": raw_output,
        "macro_tags": normalized_output["macro_tags"],
        "industry_tags": normalized_output["industry_tags"],
        "entity_tags": normalized_output["entity_tags"]
    })


Processing article 1/50...
Processing article 2/50...
Processing article 3/50...
Processing article 4/50...
Processing article 5/50...
Processing article 6/50...
Processing article 7/50...
Processing article 8/50...
Processing article 9/50...
Processing article 10/50...
Processing article 11/50...
Processing article 12/50...
Processing article 13/50...
Processing article 14/50...
Processing article 15/50...
Processing article 16/50...
Processing article 17/50...
Processing article 18/50...
Processing article 19/50...
Processing article 20/50...
Processing article 21/50...
Processing article 22/50...
Processing article 23/50...
Processing article 24/50...
Processing article 25/50...
Processing article 26/50...
Processing article 27/50...
Processing article 28/50...
Processing article 29/50...
Processing article 30/50...
Processing article 31/50...
Processing article 32/50...
Processing article 33/50...
Processing article 34/50...
Processing article 35/50...
Processing article 36/50...
P

In [42]:
results50_df = pd.DataFrame(results_50)
results50_df

,article_id,title,raw_output,macro_tags,industry_tags,entity_tags
0,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[technology],"[apple, B&H]"
1,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...,"{\n ""macro_tags"": [""inflation""],\n ""industr...",[inflation],[],"[amc, cinemark, regal_cinemas, aaa]"
2,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[technology],"[applied_intuition, embank_technology]"
3,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound...","{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],"[real_estate, financials, banking]","[Vingroup, VHM, VRE, VIC, DXG, PDR, DIG, VCG, ..."
4,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[health_care],"[Leon Cooperman, Allianz]"
5,9b09edeafdc1bb1ac6542e37f2ac3208d5094389,Wharton Professor Jeremy Siegel Says Stocks Co...,"{\n ""macro_tags"": [""fiscal_policy"", ""interes...","[fiscal_policy, interest_rates]",[financials],"[Jeremy Siegel, Federal Reserve, SPDR S&P 500 ..."
6,cb3d9ad151d0c44da42974845bcfd6e6c43223b4,HyperFiber Launches Green Team to Support Inst...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],"[technology, communication_services]",[HyperFiber]
7,b11a31f0e2cb947f5cece113461f2b927c0901c8,Forever Cheer picks Hong Kong as gateway for g...,"{\n ""macro_tags"": [""geopolitics""],\n ""indus...",[geopolitics],"[health_care, technology]","[Forever Cheer Holding, Zhuo Min, University o..."
8,f5ef318e218c131c132bccaacf7255b6e8dd6115,Chris Martin on being environmentally friendly...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[],"[chris_martin, coldplay, ellie_goulding, bbc_r..."
9,60b0393c5d31738af3082077a934927fd4824d9f,Guy Fieri Foundation donates $1.2M to Lahaina ...,"{\n ""macro_tags"": [],\n ""industry_tags"": [""...",[],[banking],"[guy_fieri, hawaii_restaurant_association, ame..."


In [43]:
test_50_labeled = test_50.merge(
    results50_df[["article_id", "macro_tags", "industry_tags", "entity_tags"]],
    on="article_id",
    how="left"
)

test_50_labeled[["article_id", "title", "macro_tags", "industry_tags", "entity_tags"]]

,article_id,title,macro_tags,industry_tags,entity_tags
0,5bce820b060ef08e7f276f84c3406ec08c6311b5,Apple's New M3 Pro MacBook Pro Gets an Epic $3...,[],[technology],"[apple, B&H]"
1,64983ee8c7ac596bcf012bc57e4466c160ebbc2c,Rossen Reports: Secret ways to save money at t...,[inflation],[],"[amc, cinemark, regal_cinemas, aaa]"
2,83d8c1441e7ac5f0ca6653117311f001e18b502c,Applied Intuition Announces Favorable Resoluti...,[],[technology],"[applied_intuition, embank_technology]"
3,2f8f94ab60b08cbb48c6a4ee55aa02b8db901de8,"Foreign investors return, stock market rebound...",[],"[real_estate, financials, banking]","[Vingroup, VHM, VRE, VIC, DXG, PDR, DIG, VCG, ..."
4,66f04e86ca2e659b12fef67f266755047efbc8df,Ground News - Sudbury's hospital draw Friday w...,[],[health_care],"[Leon Cooperman, Allianz]"
5,9b09edeafdc1bb1ac6542e37f2ac3208d5094389,Wharton Professor Jeremy Siegel Says Stocks Co...,"[fiscal_policy, interest_rates]",[financials],"[Jeremy Siegel, Federal Reserve, SPDR S&P 500 ..."
6,cb3d9ad151d0c44da42974845bcfd6e6c43223b4,HyperFiber Launches Green Team to Support Inst...,[],"[technology, communication_services]",[HyperFiber]
7,b11a31f0e2cb947f5cece113461f2b927c0901c8,Forever Cheer picks Hong Kong as gateway for g...,[geopolitics],"[health_care, technology]","[Forever Cheer Holding, Zhuo Min, University o..."
8,f5ef318e218c131c132bccaacf7255b6e8dd6115,Chris Martin on being environmentally friendly...,[],[],"[chris_martin, coldplay, ellie_goulding, bbc_r..."
9,60b0393c5d31738af3082077a934927fd4824d9f,Guy Fieri Foundation donates $1.2M to Lahaina ...,[],[banking],"[guy_fieri, hawaii_restaurant_association, ame..."


In [44]:
test_50_labeled.to_csv("silver_test_batch_50.csv", index=False)